In [ ]:
# eval_yolo

import os
import json
import shutil
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from tqdm import tqdm

from ultralytics import YOLO
from monai.metrics import DiceMetric, MeanIoU, ConfusionMatrixMetric

def evaluate_yolo_semantic(base_dir, model_path):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    test_img_dir  = os.path.join(base_dir, "images", "test")
    test_mask_dir = os.path.join(base_dir, "masks", "test")

    if not os.path.exists(test_img_dir) or not os.path.exists(test_mask_dir):
        raise FileNotFoundError(f"Diretórios de teste não encontrados em: {base_dir}")

    model_dir        = os.path.dirname(os.path.dirname(model_path))
    out_preds_dir    = os.path.join(model_dir, "preds_test")
    out_images_dir   = os.path.join(out_preds_dir, "images")
    out_overlays_dir = os.path.join(out_preds_dir, "overlays")

    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_overlays_dir, exist_ok=True)

    print(f"\n🔄 Carregando modelo YOLO...")
    print(f"📁 Caminho: {model_path}")

    try:
        model = YOLO(model_path, task='semantic')
    except Exception:
        print("Aviso: Falha ao usar task='semantic'. Tentando com o padrão de segmentação do YOLO...")
        model = YOLO(model_path, task='segment')

    # ── MÉTRICAS: reduction="mean_batch" → shape [2] por classe após aggregate ──
    monai_dice = DiceMetric(include_background=True, reduction="mean_batch")
    monai_iou  = MeanIoU(include_background=True,  reduction="mean_batch")
    monai_prec = ConfusionMatrixMetric(include_background=True, metric_name="precision", reduction="mean_batch")
    monai_rec  = ConfusionMatrixMetric(include_background=True, metric_name="recall",    reduction="mean_batch")
    monai_f1   = ConfusionMatrixMetric(include_background=True, metric_name="f1 score",  reduction="mean_batch")
    monai_acc  = ConfusionMatrixMetric(include_background=True, metric_name="accuracy",  reduction="mean_batch")

    test_images = sorted([f for f in os.listdir(test_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    print(f"\n📊 [EVAL] Lendo dados de teste da pasta: {base_dir}")
    print(f"📊 Iniciando inferência global ({len(test_images)} imagens)...")

    with torch.no_grad():
        for fname in tqdm(test_images, desc="Inference", colour='cyan'):
            img_path  = os.path.join(test_img_dir,  fname)
            mask_path = os.path.join(test_mask_dir, fname)

            orig_img_bgr = cv2.imread(img_path)
            gt_mask      = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if gt_mask is None:
                continue

            H, W = orig_img_bgr.shape[:2]

            gt_mask_bin  = (gt_mask > 0).astype(np.float32)
            targets_bin  = torch.tensor(gt_mask_bin, device=device).unsqueeze(0).unsqueeze(0)

            results = model.predict(img_path, verbose=False, imgsz=768)
            result  = results[0]

            preds_bin = torch.zeros((1, 1, H, W), device=device, dtype=torch.float32)

            if hasattr(result, 'semantic_mask') and result.semantic_mask is not None:
                mask_data = result.semantic_mask.data.cpu().numpy()
                if mask_data.shape != (H, W):
                    mask_data = cv2.resize(
                        mask_data.astype(np.uint8), (W, H),
                        interpolation=cv2.INTER_NEAREST
                    )
                preds_bin[0, 0] = torch.tensor((mask_data == 1), device=device, dtype=torch.float32)

            preds_onehot   = torch.cat([1 - preds_bin,   preds_bin],   dim=1)
            targets_onehot = torch.cat([1 - targets_bin, targets_bin], dim=1)

            monai_iou(y_pred=preds_onehot,  y=targets_onehot)
            monai_dice(y_pred=preds_onehot, y=targets_onehot)
            monai_prec(y_pred=preds_onehot, y=targets_onehot)
            monai_rec(y_pred=preds_onehot,  y=targets_onehot)
            monai_f1(y_pred=preds_onehot,   y=targets_onehot)
            monai_acc(y_pred=preds_onehot,  y=targets_onehot)

            shutil.copy2(img_path, os.path.join(out_images_dir, fname))

            pred_np     = preds_bin[0, 0].cpu().numpy().astype(bool)
            overlay_bgr = orig_img_bgr.copy()
            overlay_bgr[pred_np] = np.clip(
                overlay_bgr[pred_np] * 0.5 + np.array([0, 0, 255]) * 0.5, 0, 255
            ).astype(np.uint8)
            cv2.imwrite(os.path.join(out_overlays_dir, fname), overlay_bgr)

    # ── AGREGAÇÃO: [0] = fundo, [1] = café ──
    iou_per_class = monai_iou.aggregate()           # tensor [2]
    iou_bg        = iou_per_class[0].item()
    iou_coffee    = iou_per_class[1].item()
    miou          = iou_per_class.mean().item()

    dice_per_class = monai_dice.aggregate()         # tensor [2]
    dice_bg        = dice_per_class[0].item()
    dice_coffee    = dice_per_class[1].item()
    dice_mean      = dice_per_class.mean().item()

    prec_per_class = monai_prec.aggregate()[0]      # tensor [2]
    rec_per_class  = monai_rec.aggregate()[0]
    f1_per_class   = monai_f1.aggregate()[0]
    acc_per_class  = monai_acc.aggregate()[0]

    precision_bg     = prec_per_class[0].item()
    precision_coffee = prec_per_class[1].item()
    recall_bg        = rec_per_class[0].item()
    recall_coffee    = rec_per_class[1].item()
    f1_bg            = f1_per_class[0].item()
    f1_coffee        = f1_per_class[1].item()
    f1_macro         = f1_per_class.mean().item()   # F1 macro = média(fundo, café)
    pixel_acc        = acc_per_class.mean().item()  # média das classes

    metrics_report = {
        "Test_Samples":    len(test_images),
        "mIoU":            float(miou),
        "IoU_Fundo":       float(iou_bg),
        "IoU_Cafe":        float(iou_coffee),
        "Dice_mean":       float(dice_mean),
        "Dice_Fundo":      float(dice_bg),
        "Dice_Cafe":       float(dice_coffee),
        "F1_macro":        float(f1_macro),
        "F1_Fundo":        float(f1_bg),
        "F1_Cafe":         float(f1_coffee),
        "Precision_Fundo": float(precision_bg),
        "Precision_Cafe":  float(precision_coffee),
        "Recall_Fundo":    float(recall_bg),
        "Recall_Cafe":     float(recall_coffee),
        "PixelAcc":        float(pixel_acc)
    }

    report_path = os.path.join(out_preds_dir, "test_eval_report_yolo.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(metrics_report, f, indent=4)

    print("\n" + "="*50)
    print(" 🚀 RELATÓRIO DE AVALIAÇÃO - TESTE FINAL (YOLO SEMÂNTICO) 🚀")
    print("="*50)
    print(f" 🏆 mIoU Global          : {miou:.4f}")
    print(f" 🟤 IoU Fundo  (cls 0)   : {iou_bg:.4f}")
    print(f" 🟢 IoU Café   (cls 1)   : {iou_coffee:.4f}")
    print(f" 🎲 Dice Médio           : {dice_mean:.4f}")
    print(f" 🟤 Dice Fundo (cls 0)   : {dice_bg:.4f}")
    print(f" 🟢 Dice Café  (cls 1)   : {dice_coffee:.4f}")
    print("-"*50)
    print(f" 🎯 Precision  Fundo      : {precision_bg:.4f}")
    print(f" 🎯 Precision  Café       : {precision_coffee:.4f}")
    print(f" 🔍 Recall     Fundo      : {recall_bg:.4f}")
    print(f" 🔍 Recall     Café       : {recall_coffee:.4f}")
    print(f" ⚖️  F1 Macro  (fundo+café): {f1_macro:.4f}")
    print(f" ⚖️  F1 Fundo             : {f1_bg:.4f}")
    print(f" ⚖️  F1 Café              : {f1_coffee:.4f}")
    print(f" 📐 Pixel Acc  (média cls): {pixel_acc:.4f}")
    print("="*50)
    print(f"\n📂 Resultados salvos em: {out_preds_dir}")
    print(f"   ├── /images   (Imagens originais)")
    print(f"   └── /overlays (Máscaras sobrepostas)\n")

if __name__ == "__main__":
    DATASET_DIR    = "dataset_1024_final_cluster"
    MODEL_WEIGHTS  = "runs/semantic/resultados_cafe/semantico_cluster_final_45_1024/weights/best.pt"
    evaluate_yolo_semantic(DATASET_DIR, MODEL_WEIGHTS)
    # ==========================================


/homeLocal/victor-louzada/projeto-cafe/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔄 Carregando modelo YOLO...
📁 Caminho: runs/semantic/resultados_cafe/semantico_cluster_final_45_1024/weights/best.pt

📊 [EVAL] Lendo dados de teste da pasta: dataset_1024_final_cluster
📊 Iniciando inferência global (241 imagens)...


Inference: 100%|██████████| 241/241 [00:27<00:00,  8.72it/s]


 🚀 RELATÓRIO DE AVALIAÇÃO - TESTE FINAL (YOLO SEMÂNTICO) 🚀
 🏆 mIoU Global          : 0.8580
 🟤 IoU Fundo  (cls 0)   : 0.9482
 🟢 IoU Café   (cls 1)   : 0.7678
 🎲 Dice Médio           : 0.9196
 🟤 Dice Fundo (cls 0)   : 0.9727
 🟢 Dice Café  (cls 1)   : 0.8665
--------------------------------------------------
 🎯 Precision  Fundo      : 0.9880
 🎯 Precision  Café       : 0.8091
 🔍 Recall     Fundo      : 0.9626
 🔍 Recall     Café       : 0.9311
 ⚖️  F1 Macro  (fundo+café): 0.9205
 ⚖️  F1 Fundo             : 0.9751
 ⚖️  F1 Café              : 0.8658
 📐 Pixel Acc  (média cls): 0.9581

📂 Resultados salvos em: runs/semantic/resultados_cafe/semantico_cluster_final_45_1024/preds_test
   ├── /images   (Imagens originais)
   └── /overlays (Máscaras sobrepostas)

